# Taxonomy Hierarchical Classification - Zero-Shot with Numpy (Enhanced)

In [1]:
# 1) Setup and imports

import numpy as np
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Load Kedro context
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project

# Bootstrap the project (run from notebook directory)
project_path = Path.cwd().parent
bootstrap_project(project_path)

# Create session and get context
session = KedroSession.create(project_path=project_path)
context = session.load_context()

# Access catalog and parameters
catalog = context.catalog
params = context.params

print("Kedro context loaded successfully")
print(f"Model: {params['model_name']}")
print(f"Taxonomy: {params['taxonomy_key']}")


[01/12/26 11:53:50] INFO     Using                                                                  ]8;id=231248;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=177877;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/framework/project/__init__.py#269\269]8;;\
                             '/Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packa                
                             ges/kedro/framework/project/rich_logging.yml' as logging                              
                             configuration.                                                                        

Kedro context loaded successfully
Model: BAAI/bge-m3
Taxonomy: ISCO


In [2]:
# 2) Load taxonomy from Kedro catalog

taxonomy_dict = catalog.load("taxonomy_definition")
taxonomy_key = params["taxonomy_key"]
taxonomy = taxonomy_dict.get(taxonomy_key)()

if taxonomy is None:
    raise ValueError(f"Taxonomy '{taxonomy_key}' not found. Available: {list(taxonomy_dict.keys())}")

print(f"Loaded taxonomy: {taxonomy_key}")
print(f"Shape: {taxonomy.shape}")

# Validate and normalize
required = ["level","code","label","definition","examples","parentCode","isLeaf"]
missing = [c for c in required if c not in taxonomy.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

taxonomy = taxonomy.copy()
taxonomy["code"] = taxonomy["code"].astype(str)
taxonomy["parentCode"] = taxonomy["parentCode"].astype(str)
taxonomy["level"] = taxonomy["level"].astype(int)

if taxonomy["isLeaf"].dtype != bool:
    taxonomy["isLeaf"] = taxonomy["isLeaf"].astype(str).str.lower().isin(["true","1","yes","y"])

print(f"\nTaxonomy statistics:")
print(f"  Total nodes: {len(taxonomy)}")
print(f"  Levels: {taxonomy['level'].min()} - {taxonomy['level'].max()}")
print(f"  Leaf nodes: {taxonomy['isLeaf'].sum()}")

taxonomy.head(5)

[01/12/26 11:53:58] INFO     Loading data from taxonomy_definition (PartitionedDataset)...     ]8;id=260102;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=38603;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\

Loaded taxonomy: ISCO
Shape: (619, 8)

Taxonomy statistics:
  Total nodes: 619
  Levels: 1 - 4
  Leaf nodes: 436


,level,code,label,definition,examples,id,parentCode,isLeaf
0,1,1,Managers,"Managers plan, direct, coordinate and evaluate...",managers usually include formulating and advis...,fe6187ee,nan,False
1,2,11,"Chief Executives, Senior Officials and Legisla...","Chief executives, senior officials and legisla...",workers in this submajor group usually include...,eebe7f94,1,False
2,3,111,Legislators and Senior Officials,"Legislators and senior officials determine, fo...",presiding over or participating in the proceed...,3891a5dd,11,False
3,4,1111,Legislators,"Legislators determine, formulate, and direct p...",(a) presiding over or participating in the pr...,6a5bf9e5,111,True
4,4,1112,Senior Government Officials,Senior government officials advise governments...,"(a) advising national, state, regional or loc...",ed2a37c5,111,True


In [3]:
# 3) Build taxonomy adjacency (parent-child relationships)

code_to_row: Dict[str, dict] = taxonomy.set_index("code").to_dict(orient="index")

children: Dict[str, List[str]] = {code: [] for code in taxonomy["code"]}
parent: Dict[str, str] = {}

for _, row in taxonomy.iterrows():
    code = str(row["code"])
    pcode = str(row["parentCode"]) if pd.notna(row["parentCode"]) else ""
    if pcode and pcode in code_to_row:
        parent[code] = pcode
        children[pcode].append(code)

roots = [code for code in taxonomy["code"] if code not in parent]
print(f"Roots (forest size): {len(roots)}")
print("Example roots:", roots[:10])

# Helper functions
def is_ancestor_or_equal(a: str, b: str) -> bool:
    """Check if a is ancestor of b (or a == b)."""
    cur = b
    if a == b:
        return True
    while cur in parent:
        cur = parent[cur]
        if cur == a:
            return True
    return False

def ancestors_of(nodes: Set[str]) -> Set[str]:
    """Get all ancestors of a set of nodes."""
    out: Set[str] = set()
    for n in nodes:
        cur = n
        while cur in parent:
            cur = parent[cur]
            if cur in out:
                break
            out.add(cur)
    return out

def children_of(nodes: Set[str]) -> Set[str]:
    """Get all children of a set of nodes."""
    out: Set[str] = set()
    for n in nodes:
        out.update(children.get(n, []))
    return out

def roots_of(nodes: Set[str]) -> Set[str]:
    """Find root nodes for a set of candidates."""
    out: Set[str] = set()
    for n in nodes:
        cur = n
        while cur in parent:
            cur = parent[cur]
        out.add(cur)
    return out

print(f"\n✅ Taxonomy graph built")

Roots (forest size): 10
Example roots: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '0']

✅ Taxonomy graph built


In [4]:
# 4) Build multi-view embedding index with sentence-transformers + numpy
#
# Creates separate embeddings for:
# - E_label: Always present (used for retrieval)
# - E_definition: Only if definition exists (used for re-ranking)
# - E_examples: Only if examples exist (used for re-ranking)

MODEL = params["model_name"]
zero_shot_config = params.get("zero_shot", {})

print(f"Loading embedding model: {MODEL}")
model = SentenceTransformer(MODEL)

# Get embedding dimension
test_embed = model.encode(["test"])
embedding_dim = test_embed.shape[1]
print(f"Embedding dimension: {embedding_dim}")

# Create code-to-index mapping (for fast lookup)
all_codes = taxonomy["code"].tolist()
code_to_idx = {code: idx for idx, code in enumerate(all_codes)}
idx_to_code = {idx: code for code, idx in code_to_idx.items()}

# === Label embeddings (always present, used for retrieval) ===
all_labels = [str(code_to_row[code]["label"]) for code in all_codes]

print(f"\nEmbedding {len(all_labels)} taxonomy labels...")
label_embeddings = model.encode(
    all_labels,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # L2 normalization for cosine similarity
)

print(f"Label embeddings shape: {label_embeddings.shape}")

# === Definition embeddings (only where present) ===
def_mask = taxonomy["definition"].notna()
def_count = def_mask.sum()

if def_count > 0:
    print(f"\nEmbedding {def_count} taxonomy definitions...")
    def_texts = taxonomy.loc[def_mask, "definition"].fillna("").tolist()
    def_embeddings_temp = model.encode(
        def_texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Create sparse storage (only for nodes with definitions)
    def_codes_with_def = taxonomy.loc[def_mask, "code"].tolist()
    def_code_to_idx = {code: i for i, code in enumerate(def_codes_with_def)}
    def_embeddings = def_embeddings_temp
    print(f"Definition embeddings shape: {def_embeddings.shape}")
else:
    def_code_to_idx = {}
    def_embeddings = None
    print("\nNo definitions found, skipping definition embeddings")

# === Example embeddings (only where present) ===
ex_mask = taxonomy["examples"].notna()
ex_count = ex_mask.sum()

if ex_count > 0:
    print(f"\nEmbedding {ex_count} taxonomy examples...")
    ex_texts = taxonomy.loc[ex_mask, "examples"].fillna("").tolist()
    ex_embeddings_temp = model.encode(
        ex_texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Create sparse storage (only for nodes with examples)
    ex_codes_with_ex = taxonomy.loc[ex_mask, "code"].tolist()
    ex_code_to_idx = {code: i for i, code in enumerate(ex_codes_with_ex)}
    ex_embeddings = ex_embeddings_temp
    print(f"Example embeddings shape: {ex_embeddings.shape}")
else:
    ex_code_to_idx = {}
    ex_embeddings = None
    print("\nNo examples found, skipping example embeddings")

print(f"\n✅ Multi-view embedding index built")
print(f"   - {len(all_codes)} nodes")
print(f"   - {embedding_dim} dimensions")
print(f"   - Label embeddings: {len(all_codes)} (100%)")
print(f"   - Definition embeddings: {def_count} ({def_count/len(all_codes)*100:.1f}%)")
print(f"   - Example embeddings: {ex_count} ({ex_count/len(all_codes)*100:.1f}%)")
print(f"   - Normalized for cosine similarity")

Loading embedding model: BAAI/bge-m3
Embedding dimension: 1024

Embedding 619 taxonomy labels...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Label embeddings shape: (619, 1024)

Embedding 619 taxonomy definitions...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Definition embeddings shape: (619, 1024)

Embedding 619 taxonomy examples...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Example embeddings shape: (619, 1024)

✅ Multi-view embedding index built
   - 619 nodes
   - 1024 dimensions
   - Label embeddings: 619 (100%)
   - Definition embeddings: 619 (100.0%)
   - Example embeddings: 619 (100.0%)
   - Normalized for cosine similarity


In [5]:
def ancestors_including_self(code: str, parent: dict) -> list[str]:
    """
    Returns [code, parent(code), parent(parent(code)), ...] up to root.
    Root is identified by missing/None parent.
    """
    out = []
    cur = code
    seen = set()
    while cur is not None and cur not in seen:
        out.append(cur)
        seen.add(cur)
        cur = parent.get(cur)
    return out


def path_to_root(code: str, parent: dict) -> list[str]:
    """
    Returns [root, ..., code].
    """
    anc = ancestors_including_self(code, parent)
    return list(reversed(anc))


def ancestor_at_level(code: str, target_level: int, parent: dict, level: dict) -> str | None:
    """
    Walk up until node is at target_level. Returns None if cannot reach.
    """
    cur = code
    seen = set()
    while cur is not None and cur not in seen:
        seen.add(cur)
        if level.get(cur) == target_level:
            return cur
        # if we went above target level (smaller number), stop
        if level.get(cur, 10**9) < target_level:
            return None
        cur = parent.get(cur)
    return None

In [15]:
# 5) Global Retrieval + Candidate Set Construction (Top-K at any level)

K_RETRIEVAL = int(params.get("top_k", 20))  # default 20
M_TOP = int(params.get("top_m", 15))        # top-M for stopping decision
STOP_MODE = params.get("stop_mode", "topm") # "topm" or "lca"

print(f"Retrieval config: K={K_RETRIEVAL}, M={M_TOP}, STOP_MODE={STOP_MODE}")

# Embed query
query = params.get("query_text", "Legislators")
qvec = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]

# ANN retrieval over label embeddings
# label_embeddings: numpy array aligned with taxonomy['code'] in code_to_idx
sims_all = label_embeddings @ qvec  # dot == cosine because normalized
top_idx = np.argsort(-sims_all)[:K_RETRIEVAL]
retrieved = [(idx_to_code[i], float(sims_all[i])) for i in top_idx]

print("\n" + "="*80)
print(f"RETRIEVAL FOR QUERY: '{query}'")
print("="*80)
for rank,(code,sim) in enumerate(retrieved[:20], start=1):
    row = code_to_row[code]
    print(f"{rank:>3}. {code:6s} | L{row['level']} | {row['label'][:45]:45s} → sim={sim:.3f}")

retrieved_codes = [c for c,_ in retrieved]
retrieved_set = set(retrieved_codes)

# Candidate nodes = retrieved + all their ancestors
cand_nodes = set()
for c in retrieved_codes:
    cand_nodes.update(ancestors_including_self(c, parent))

print(f"\nCandidate nodes (retrieved + ancestors): {len(cand_nodes)}")

Retrieval config: K=20, M=15, STOP_MODE=topm

RETRIEVAL FOR QUERY: 'Legislators'
  1. 1111   | L4 | Legislators                                   → sim=1.000
  2. 111    | L3 | Legislators and Senior Officials              → sim=0.767
  3. 11     | L2 | Chief Executives, Senior Officials and Legisl → sim=0.727
  4. 2611   | L4 | Lawyers                                       → sim=0.712
  5. 2612   | L4 | Judges                                        → sim=0.677
  6. 2261   | L4 | Dentists                                      → sim=0.675
  7. 2642   | L4 | Journalists                                   → sim=0.661
  8. 3342   | L4 | Legal Secretaries                             → sim=0.661
  9. 82     | L2 | Assemblers                                    → sim=0.661
 10. 821    | L3 | Assemblers                                    → sim=0.661
 11. 2631   | L4 | Economists                                    → sim=0.660
 12. 261    | L3 | Legal Professionals                           → sim=0

In [21]:
# 5b) Multi-View Re-Ranking
#
# After retrieving top-K candidates using label embeddings, re-rank them using
# multi-view scoring to boost precision:
#
# score(n) = max(
#     qvec @ label[n],
#     qvec @ definition[n]   if definition exists else -inf,
#     qvec @ examples[n]     if examples exist else -inf
# )
#
# This allows nodes with strong semantic matches in definitions or examples
# to rank higher, even if their labels are less similar.

def compute_multiview_score(code: str, qvec: np.ndarray) -> Tuple[float, str]:
    """
    Compute multi-view max score for a given code.
    
    Returns:
        (max_score, best_view) where best_view is one of: 'label', 'definition', 'examples'
    """
    # Label score (always present)
    label_sim = float(label_embeddings[code_to_idx[code]] @ qvec)
    
    # Definition score (if exists)
    if code in def_code_to_idx:
        def_sim = float(def_embeddings[def_code_to_idx[code]] @ qvec)
    else:
        def_sim = -np.inf
    
    # Example score (if exists)
    if code in ex_code_to_idx:
        ex_sim = float(ex_embeddings[ex_code_to_idx[code]] @ qvec)
    else:
        ex_sim = -np.inf
    
    # Return max score and which view achieved it
    scores = [
        (label_sim, 'label'),
        (def_sim, 'definition'),
        (ex_sim, 'examples')
    ]
    max_score, best_view = max(scores, key=lambda x: x[0])
    
    return max_score, best_view

# Re-rank retrieved candidates using multi-view scoring
retrieved_reranked = []
for code, original_sim in retrieved:
    multiview_score, best_view = compute_multiview_score(code, qvec)
    retrieved_reranked.append((code, multiview_score, original_sim, best_view))

# Sort by multi-view score (descending)
retrieved_reranked.sort(key=lambda x: x[1], reverse=True)

# Update retrieved list with re-ranked results (keeping original format for compatibility)
retrieved = [(code, mv_score) for code, mv_score, _, _ in retrieved_reranked]
retrieved_codes = [c for c, _ in retrieved]

print("\n" + "="*80)
print("MULTI-VIEW RE-RANKING RESULTS")
print("="*80)
print(f"Top-{min(20, len(retrieved_reranked))} after re-ranking:\n")
for rank, (code, mv_score, orig_sim, best_view) in enumerate(retrieved_reranked[:20], start=1):
    row = code_to_row[code]
    boost = mv_score - orig_sim
    boost_marker = f"↑{boost:+.3f}" if boost > 0.01 else ""
    view_marker = f"[{best_view[0].upper()}]" if best_view != 'label' else ""
    print(f"{rank:>3}. {code:6s} | L{row['level']} | {row['label'][:40]:40s} → {mv_score:.3f} {view_marker:3s} {boost_marker}")

print(f"\nRe-ranking statistics:")
label_best = sum(1 for _, _, _, view in retrieved_reranked if view == 'label')
def_best = sum(1 for _, _, _, view in retrieved_reranked if view == 'definition')
ex_best = sum(1 for _, _, _, view in retrieved_reranked if view == 'examples')
print(f"  Best view = label:      {label_best}/{len(retrieved_reranked)} ({label_best/len(retrieved_reranked)*100:.1f}%)")
print(f"  Best view = definition: {def_best}/{len(retrieved_reranked)} ({def_best/len(retrieved_reranked)*100:.1f}%)")
print(f"  Best view = examples:   {ex_best}/{len(retrieved_reranked)} ({ex_best/len(retrieved_reranked)*100:.1f}%)")

# Rebuild candidate nodes with re-ranked retrieved set
cand_nodes = set()
for c in retrieved_codes:
    cand_nodes.update(ancestors_including_self(c, parent))

print(f"\nCandidate nodes (re-ranked retrieved + ancestors): {len(cand_nodes)}")



MULTI-VIEW RE-RANKING RESULTS
Top-20 after re-ranking:

  1. 1111   | L4 | Legislators                              → 1.000     
  2. 111    | L3 | Legislators and Senior Officials         → 0.767     
  3. 11     | L2 | Chief Executives, Senior Officials and L → 0.727     
  4. 2611   | L4 | Lawyers                                  → 0.712     
  5. 2612   | L4 | Judges                                   → 0.677     
  6. 2261   | L4 | Dentists                                 → 0.675     
  7. 2642   | L4 | Journalists                              → 0.661     
  8. 3342   | L4 | Legal Secretaries                        → 0.661     
  9. 82     | L2 | Assemblers                               → 0.661     
 10. 821    | L3 | Assemblers                               → 0.661     
 11. 2631   | L4 | Economists                               → 0.660     
 12. 261    | L3 | Legal Professionals                      → 0.650     
 13. 2262   | L4 | Pharmacists                              → 0.646

In [40]:

# 6) Ancestor Support (Votes) + Path Scoring (Support × Cosine along path)
#
# Weighted support:
# - Each retrieved hit contributes weight w = max(sim, 0)
# - That weight is added to the node itself and ALL its ancestors
# - Normalized so support[a] in [0,1]
#
# Path score for candidate n:
#   ScorePath(n) = Σ_{a in path(root→n)} gamma^i * support[a] * sim_multiview[a]


GAMMA = float(params.get("gamma", 0.75))

# Build weighted support using re-ranked multi-view scores
support_raw = defaultdict(float)
total_w = 0.0
for code, sim in retrieved:  # retrieved now contains multi-view scores from Cell 7
    w = max(sim, 0.0)
    total_w += w
    for a in ancestors_including_self(code, parent):
        support_raw[a] += w

support = {}
den = total_w if total_w > 0 else 1.0
for a, v in support_raw.items():
    support[a] = v / den

# Precompute multi-view sims for ALL candidate nodes (not just retrieved)
# For candidates that weren't in top-K retrieval, compute their multi-view score now
#cand_list = sorted(cand_nodes)
cand_list = sorted(set([c for c,_ in retrieved]) | set().union(*[ancestors_including_self(c,parent) for c,_ in retrieved]))
sim_multiview = {}

for code in cand_list:
    # Check if this code was in the retrieved set (already has multi-view score)
    retrieved_dict = {c: score for c, score in retrieved}
    if code in retrieved_dict:
        sim_multiview[code] = retrieved_dict[code]
    else:
        # This is an ancestor node not in retrieved set, compute its multi-view score
        mv_score, _ = compute_multiview_score(code, qvec)
        sim_multiview[code] = mv_score

def path_score(code: str, gamma: float = GAMMA) -> float:
    path = path_to_root(code, parent)
    s = 0.0
    for i, a in enumerate(path):
        s += (gamma ** len(path)-i) * support.get(a, 0.0) * sim_multiview.get(a, -1.0)
    return float(s)

# Rank candidates by path score
ranked = sorted(cand_list, key=path_score, reverse=True)
topM = ranked[:M_TOP]

print("\n" + "="*80)
print("TOP SUPPORT NODES (by support)")
print("="*80)
top_support = sorted(support.items(), key=lambda x: x[1], reverse=True)[:12]
for code, sup in top_support:
    row = code_to_row[code]
    print(f"{code:6s} | L{row['level']} | {row['label'][:45]:45s} → support={sup:.3f}, sim_mv={sim_multiview.get(code, -1):.3f}")

print("\n" + "="*80)
print(f"TOP-{M_TOP} CANDIDATES (by path_score with multi-view)")
print("="*80)
for i, code in enumerate(topM, start=1):
    row = code_to_row[code]
    ps = path_score(code)
    print(f"{i:>2}. {code:6s} | L{row['level']} | {row['label'][:45]:45s} → path_score={ps:.4f}  (sim_mv={sim_multiview[code]:.3f}, support={support.get(code,0):.3f})")



TOP SUPPORT NODES (by support)
2      | L1 | Professionals                                 → support=0.532, sim_mv=0.635
26     | L2 | Legal, Social and Cultural Professionals      → support=0.294, sim_mv=0.517
11     | L2 | Chief Executives, Senior Officials and Legisl → support=0.230, sim_mv=0.727
1      | L1 | Managers                                      → support=0.230, sim_mv=0.615
111    | L3 | Legislators and Senior Officials              → support=0.176, sim_mv=0.767
261    | L3 | Legal Professionals                           → support=0.150, sim_mv=0.650
82     | L2 | Assemblers                                    → support=0.097, sim_mv=0.661
8      | L1 | Plant and Machine Operators, and Assemblers   → support=0.097, sim_mv=0.505
226    | L3 | Other Health Professionals                    → support=0.097, sim_mv=0.572
22     | L2 | Health Professionals                          → support=0.097, sim_mv=0.573
3      | L1 | Technicians and Associate Professionals       → suppor

In [39]:

# 7) Depth Decision / Stopping (Selectable: LCA or Top-M Concentration)
#
# Problem solved:
# - Over-specification: If top candidates are near-tied siblings (e.g., Education subtypes),
#   return their shared ancestor instead of randomly choosing a child.
#
# Two modes:
# - STOP_MODE="lca": compute LCA(topM); if best doesn't dominate, return LCA
# - STOP_MODE="topm": compute consensus by level over topM; descend while consensus holds; stop when it collapses

DOMINANCE_DELTA = float(params.get("dominance_delta", 0.05))

def lowest_common_ancestor(nodes: List[str]) -> str:
    if not nodes:
        return None
    # ancestor set of first
    common = set(ancestors_including_self(nodes[0], parent))
    for n in nodes[1:]:
        common &= set(ancestors_including_self(n, parent))
        if not common:
            return None
    # choose deepest common node (max level)
    return max(common, key=lambda c: code_to_row[c]["level"])

def ancestor_at_level(code: str, target_level: int) -> str:
    cur = code
    while cur is not None and code_to_row[cur]["level"] > target_level:
        cur = parent.get(cur)
    if cur is None:
        return None
    return cur if code_to_row[cur]["level"] == target_level else None

def choose_with_lca(top_nodes: List[str]) -> Dict:
    best = top_nodes[0]
    lca = lowest_common_ancestor(top_nodes)
    if lca is None:
        return {"final": best, "reason": "no_lca", "lca": None}
    best_s = path_score(best)
    runner_s = path_score(top_nodes[1]) if len(top_nodes) > 1 else -1e9
    if (best_s - runner_s) < DOMINANCE_DELTA:
        return {"final": lca, "reason": "lca_tie_stop", "lca": lca}
    return {"final": best, "reason": "best_dominates", "lca": lca}

def level_mass(top_nodes: List[str], target_level: int) -> Dict[str, float]:
    mass = defaultdict(float)
    for n in top_nodes:
        a = ancestor_at_level(n, target_level)
        if a is not None:
            mass[a] += path_score(n)  # mass from node evidence
    total = sum(mass.values()) or 1.0
    for k in list(mass.keys()):
        mass[k] = mass[k] / total
    return dict(mass)

P1_MIN = float(params.get("p1_min", 0.60))
GAP_MIN = float(params.get("gap_min", 0.10))

def choose_with_topm_concentration(top_nodes: List[str]) -> Dict:
    # Determine max taxonomy level present
    maxL = int(max(code_to_row[n]["level"] for n in top_nodes))
    chosen = None
    trace = []
    for L in range(1, maxL + 1):
        mass = level_mass(top_nodes, L)
        if not mass:
            break
        ranked_mass = sorted(mass.items(), key=lambda x: x[1], reverse=True)
        (n1, p1) = ranked_mass[0]
        p2 = ranked_mass[1][1] if len(ranked_mass) > 1 else 0.0
        gap = p1 - p2
        trace.append((L, n1, p1, p2, gap))
        if p1 >= P1_MIN and gap >= GAP_MIN:
            chosen = n1
            continue
        # ambiguity begins here -> stop at previous chosen
        return {"final": chosen if chosen else n1, "reason": f"ambiguity_level_{L}", "trace": trace}
    return {"final": chosen if chosen else top_nodes[0], "reason": "max_consensus_depth", "trace": trace}

# Execute stopping decision
if STOP_MODE.lower() == "lca":
    decision = choose_with_lca(topM)
else:
    decision = choose_with_topm_concentration(topM)

final_code = decision["final"]
final_row = code_to_row[final_code]

print("\n" + "="*80)
print("FINAL DECISION (Consensus-based depth selection)")
print("="*80)
print(f"Final: {final_code} | L{final_row['level']} | {final_row['label']}")
print(f"Reason: {decision.get('reason')}")

if STOP_MODE.lower() == "lca":
    lca = decision.get("lca")
    if lca:
        lr = code_to_row[lca]
        print(f"LCA(topM): {lca} | L{lr['level']} | {lr['label']}")
else:
    print("\nConsensus trace by level:")
    for (L, n1, p1, p2, gap) in decision.get("trace", []):
        r = code_to_row[n1]
        print(f"  L{L}: {n1} | {r['label'][:40]:40s} p1={p1:.2f} p2={p2:.2f} gap={gap:.2f}")



FINAL DECISION (Consensus-based depth selection)
Final: 2 | L1 | Professionals
Reason: ambiguity_level_2

Consensus trace by level:
  L1: 2 | Professionals                            p1=0.84 p2=0.08 gap=0.77
  L2: 25 | Information and Communications Technolog p1=0.30 p2=0.29 gap=0.00


In [54]:
# HiRAG-STYLE SCOPED VALIDATION (NO expansion) — uses reranked pool (topM) + evidence mass
#
# Changes vs previous:
# - Validation pool = topM (reranked candidates), not raw retrieved results
# - L* and L_sub selected by path_score within pool
# - Add mass-based guard to prevent "agreeing too often"
# - Optional: allow "DEEPER" within subtree if evidence strongly prefers a descendant

OVERRIDE_MARGIN = float(params.get("hirag_override_margin", 0.10))
STABILITY_MARGIN = float(params.get("hirag_stability_margin", 0.08))

# Evidence-mass thresholds (new)
MASS_IN_MIN = float(params.get("hirag_mass_in_min", 0.55))          # need >=55% of mass inside TD to call CONSISTENT
MASS_OUT_RATIO = float(params.get("hirag_mass_out_ratio", 1.30))    # if outside mass is 1.3x inside, treat as conflict/override

# Optional "deepen within subtree" (off by default)
ALLOW_DEEPER = bool(params.get("hirag_allow_deeper", False))
DEEPER_MARGIN = float(params.get("hirag_deeper_margin", 0.08))

def is_ancestor_or_equal(a: str, b: str, parent: dict) -> bool:
    cur = b
    seen = set()
    while cur is not None and cur not in seen:
        if cur == a:
            return True
        seen.add(cur)
        cur = parent.get(cur)
    return False

def best_two(codes: list[str]):
    """Return (best_code, best_score, second_code, second_score) by path_score."""
    if not codes:
        return None, -1e9, None, -1e9
    ranked = sorted(((c, float(path_score(c))) for c in codes), key=lambda x: x[1], reverse=True)
    best_c, best_s = ranked[0]
    if len(ranked) > 1:
        sec_c, sec_s = ranked[1]
    else:
        sec_c, sec_s = None, -1e9
    return best_c, best_s, sec_c, sec_s

def pool_mass(codes: list[str]) -> float:
    """Total evidence mass over a set using path_score (scoped)."""
    return float(sum(max(path_score(c), 0.0) for c in codes))

TD = final_code

# --- Validation pool: use reranked topM (NOT raw retrieved) ---
pool_codes = list(topM) if "topM" in globals() and topM else []
pool_kind = "topM"

# Prefer leaves if present
pool_leaves = [c for c in pool_codes if bool(code_to_row[c]["isLeaf"])]
if pool_leaves:
    pool = pool_leaves
    pool_kind = "topM_leaf"
else:
    pool = pool_codes
    pool_kind = "topM_node"

# If pool is empty, fallback to retrieved codes (still better than nothing)
if not pool:
    retrieved_codes = [c for (c, _) in retrieved]
    retrieved_leaves = [c for c in retrieved_codes if bool(code_to_row[c]["isLeaf"])]
    pool = retrieved_leaves if retrieved_leaves else retrieved_codes
    pool_kind = "retrieved_leaf" if retrieved_leaves else "retrieved_node"

L_star, s_star, L2, s2 = best_two(pool)

# best evidence under TD subtree (scoped)
sub_pool = [c for c in pool if is_ancestor_or_equal(TD, c, parent)]
L_sub, s_sub, _, _ = best_two(sub_pool)

# guards
in_subtree = (L_star is not None) and is_ancestor_or_equal(TD, L_star, parent)
stability = s_star - s2 if L2 is not None else 1e9
margin = (s_star - s_sub) if (L_sub is not None) else None

# --- NEW: mass-based guard ---
mass_total = pool_mass(pool) if pool else 0.0
mass_in = pool_mass(sub_pool) if sub_pool else 0.0
mass_out = max(mass_total - mass_in, 0.0)

mass_in_ratio = (mass_in / mass_total) if mass_total > 0 else 0.0
mass_out_ratio = (mass_out / mass_in) if mass_in > 0 else float("inf")

status = "WEAK_VALIDATION"
decision_scoped = TD
reason = ""

if L_star is None:
    status = "WEAK_VALIDATION"
    reason = "no_pool_evidence"
elif in_subtree:
    # Only CONSISTENT if evidence mass supports TD subtree sufficiently
    if mass_in_ratio >= MASS_IN_MIN:
        status = "CONSISTENT"
        reason = f"best_{pool_kind}_within_TD_and_mass_in_ratio={mass_in_ratio:.2f}"
    else:
        status = "WEAK_VALIDATION"
        reason = f"best_{pool_kind}_within_TD_but_low_mass_in_ratio={mass_in_ratio:.2f}"
else:
    # best evidence is outside TD subtree
    if L_sub is None:
        status = "CONFLICT"
        reason = f"best_{pool_kind}_outside_TD_and_no_in_subtree_evidence"
    else:
        # Override only if (a) decisive margin+stability AND (b) outside mass dominates
        decisive = (margin is not None) and (margin >= OVERRIDE_MARGIN) and (stability >= STABILITY_MARGIN)
        outside_dominates = (mass_out_ratio >= MASS_OUT_RATIO)

        if decisive and outside_dominates:
            status = "OVERRIDE"
            decision_scoped = L_star
            reason = (
                f"outside_best_decisive (margin={margin:.3f}, stability={stability:.3f}, "
                f"mass_out/in={mass_out_ratio:.2f})"
            )
        else:
            status = "CONFLICT"
            reason = (
                f"outside_best_not_decisive (margin={margin:.3f}, stability={stability:.3f}, "
                f"mass_in_ratio={mass_in_ratio:.2f}, mass_out/in={mass_out_ratio:.2f})"
            )

# Optional: allow deepening within subtree when evidence strongly prefers a descendant of TD
if status in ("CONSISTENT", "WEAK_VALIDATION") and ALLOW_DEEPER and L_sub is not None:
    # If the best leaf under TD is meaningfully stronger than TD itself, suggest deeper
    td_score = float(path_score(TD))
    if (s_sub - td_score) >= DEEPER_MARGIN and is_ancestor_or_equal(TD, L_sub, parent) and (L_sub != TD):
        status = "DEEPER"
        decision_scoped = L_sub
        reason = f"deepen_within_TD (sub-td margin={s_sub-td_score:.3f})"

print("\n" + "="*80)
print("HiRAG-STYLE SCOPED VALIDATION (NO expansion, evidence=topM rerank)")
print("="*80)
print(f"TD (final_code): {TD} | L{code_to_row[TD]['level']} | {code_to_row[TD]['label']}")
print(f"Pool kind: {pool_kind} | pool size={len(pool)}")
print(f"Mass in TD subtree: {mass_in:.4f} / total {mass_total:.4f} => in_ratio={mass_in_ratio:.2f}, out/in={mass_out_ratio:.2f}")
print(f"Decision: {decision_scoped} | Status: {status} | Reason: {reason}")

if L_star:
    print(f"L* (best in pool): {L_star} | L{code_to_row[L_star]['level']} | {code_to_row[L_star]['label']} | path_score={s_star:.4f}")
if L2:
    print(f"L2 (2nd best in pool): {L2} | path_score={s2:.4f} | stability={stability:.4f}")
if L_sub:
    print(f"L_sub (best under TD): {L_sub} | L{code_to_row[L_sub]['level']} | {code_to_row[L_sub]['label']} | path_score={s_sub:.4f}")
else:
    print("L_sub: None (no pool evidence inside TD subtree)")


HiRAG-STYLE SCOPED VALIDATION (NO expansion, evidence=topM rerank)
TD (final_code): 2 | L1 | Professionals
Pool kind: topM_leaf | pool size=8
Mass in TD subtree: 4.2078 / total 4.2078 => in_ratio=1.00, out/in=0.00
Decision: 2 | Status: CONSISTENT | Reason: best_topM_leaf_within_TD_and_mass_in_ratio=1.00
L* (best in pool): 2611 | L4 | Lawyers | path_score=0.6027
L2 (2nd best in pool): 2612 | path_score=0.6007 | stability=0.0020
L_sub (best under TD): 2611 | L4 | Lawyers | path_score=0.6027


In [52]:

# 9) Summary Output (Explainability)
#
# Provide:
# - Retrieval top hits
# - Support top nodes
# - TopM by path score
# - Final decision + reason
# - Scoped validation status

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Query: {query}")
print(f"STOP_MODE: {STOP_MODE} | K={K_RETRIEVAL} | M={M_TOP} | gamma={GAMMA}")
print(f"Prediction: {final_code} | L{code_to_row[final_code]['level']} | {code_to_row[final_code]['label']}")
print(f"Reason: {decision.get('reason')}")


SUMMARY
Query: Legislators
STOP_MODE: lca | K=15 | M=15 | gamma=0.75
Prediction: 2 | L1 | Professionals
Reason: lca_tie_stop


# 10) Validation Against Training Data

Now we validate the algorithm against labeled training data from the catalog:
- **taxonomy_training**: Training data with taxonomy hierarchies as text
- **training_sentences**: Structured training sentences with annotations at all levels

This validation will:
1. Load both datasets for the current taxonomy
2. Run the classification algorithm on training queries from both datasets
3. Compare predictions against ground truth labels
4. Report accuracy metrics at each hierarchy level for both datasets

In [24]:
# 10a) Load Training Data

import json
from typing import List, Dict, Tuple
from collections import defaultdict

# Load taxonomy_training (CSV format)
print("Loading taxonomy_training dataset...")
taxonomy_training_dict = catalog.load("taxonomy_training")
taxonomy_training_df = taxonomy_training_dict.get(taxonomy_key)()

print(f"Loaded {len(taxonomy_training_df)} training examples from taxonomy_training")
print(f"Columns: {list(taxonomy_training_df.columns)}")

# Load training_sentences (JSON format)
print("\nLoading training_sentences dataset...")
training_sentences_dict = catalog.load("training_sentences")
training_sentences_data = training_sentences_dict.get(taxonomy_key)()

print(f"Loaded training_sentences for taxonomy: {training_sentences_data['taxonomyKey']}")
print(f"Number of annotated sentences: {len(training_sentences_data['sentences'])}")

# Display sample from each dataset
print("\n" + "="*80)
print("SAMPLE FROM TAXONOMY_TRAINING")
print("="*80)
print(taxonomy_training_df.head(3))

print("\n" + "="*80)
print("SAMPLE FROM TRAINING_SENTENCES")
print("="*80)
for i, sent in enumerate(training_sentences_data['sentences'][:3]):
    print(f"\nSentence {i+1}:")
    print(f"  ID: {sent['sentenceId']}")
    print(f"  Fields: {sent['fields']}")
    print(f"  Annotations: {sent['annotations']}")

# Choose which dataset to validate against
validation_data = training_sentences_data['sentences']
print(f"\n✅ Datasets loaded successfully")


Loading taxonomy_training dataset...


[01/12/26 11:13:27] INFO     Loading data from taxonomy_training (PartitionedDataset)...       ]8;id=449121;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=286429;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\

Loaded 611 training examples from taxonomy_training
Columns: ['text', 'label', 'code', 'parent_code', 'level', 'taxonomyKey']

Loading training_sentences dataset...


                    INFO     Loading data from training_sentences (PartitionedDataset)...      ]8;id=390162;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=704259;file:///Users/gabriele/anaconda3/envs/taxomind-env/lib/python3.13/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\

Loaded training_sentences for taxonomy: ISCO
Number of annotated sentences: 11803

SAMPLE FROM TAXONOMY_TRAINING
                                                text  \
0  Managers plan, direct, coordinate and evaluate...   
1  Chief executives, senior officials and legisla...   
2  Legislators and senior officials determine, fo...   

                                               label  code  parent_code  \
0                                           Managers     1          NaN   
1  Chief Executives, Senior Officials and Legisla...    11          1.0   
2                   Legislators and Senior Officials   111         11.0   

   level taxonomyKey  
0      1        ISCO  
1      2        ISCO  
2      3        ISCO  

SAMPLE FROM TRAINING_SENTENCES

Sentence 1:
  ID: 3b83c79f535d489781b523a2
  Fields: {'Job Description': 'baby sitter', 'Industry Description': 'caring for baby'}
  Annotations: [{'level': 1, 'code': '5'}, {'level': 2, 'code': '53'}, {'level': 3, 'code': '531'}, {'lev

In [25]:
# 10b) Define Classification Function

def classify_query(
    query_text: str,
    k_retrieval: int = K_RETRIEVAL,
    m_top: int = M_TOP,
    gamma: float = GAMMA,
    stop_mode: str = STOP_MODE,
    verbose: bool = False
) -> Dict:
    """
    Run the full classification pipeline on a query with multi-view re-ranking.
    
    Returns:
        dict with keys: final_code, final_level, reason, retrieved, topM, decision
    """
    # Embed query
    qvec = model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)[0]
    
    # STEP 1: Retrieval using label embeddings
    sims_all = label_embeddings @ qvec
    top_idx = np.argsort(-sims_all)[:k_retrieval]
    retrieved_initial = [(idx_to_code[i], float(sims_all[i])) for i in top_idx]
    
    # STEP 1b: Multi-view re-ranking
    retrieved_reranked = []
    for code, original_sim in retrieved_initial:
        multiview_score, best_view = compute_multiview_score(code, qvec)
        retrieved_reranked.append((code, multiview_score, original_sim, best_view))
    
    # Sort by multi-view score
    retrieved_reranked.sort(key=lambda x: x[1], reverse=True)
    
    # Update retrieved list with re-ranked scores
    retrieved = [(code, mv_score) for code, mv_score, _, _ in retrieved_reranked]
    retrieved_codes = [c for c, _ in retrieved]
    
    # Build candidate set
    cand_nodes = set()
    for c in retrieved_codes:
        cand_nodes.update(ancestors_including_self(c, parent))
    
    # STEP 2: Weighted support (using re-ranked multi-view scores)
    support_raw = defaultdict(float)
    total_w = 0.0
    for code, sim in retrieved:
        w = max(sim, 0.0)
        total_w += w
        for a in ancestors_including_self(code, parent):
            support_raw[a] += w
    
    support = {}
    den = total_w if total_w > 0 else 1.0
    for a, v in support_raw.items():
        support[a] = v / den
    
    # Precompute multi-view sims for ALL candidate nodes
    cand_list = sorted(cand_nodes)
    sim_multiview = {}
    retrieved_dict = {c: score for c, score in retrieved}
    
    for code in cand_list:
        if code in retrieved_dict:
            sim_multiview[code] = retrieved_dict[code]
        else:
            # Ancestor node not in retrieved set, compute its multi-view score
            mv_score, _ = compute_multiview_score(code, qvec)
            sim_multiview[code] = mv_score
    
    # STEP 3: Path scoring using multi-view scores
    def compute_path_score(code: str) -> float:
        path = path_to_root(code, parent)
        s = 0.0
        for i, a in enumerate(path):
            s += (gamma ** i) * support.get(a, 0.0) * sim_multiview.get(a, -1.0)
        return float(s)
    
    ranked = sorted(cand_list, key=compute_path_score, reverse=True)
    topM = ranked[:m_top]
    
    # STEP 4: Stopping decision
    if stop_mode.lower() == "lca":
        decision = choose_with_lca(topM)
    else:
        # Use topm concentration
        maxL = int(max(code_to_row[n]["level"] for n in topM))
        chosen = None
        trace = []
        for L in range(1, maxL + 1):
            mass = level_mass(topM, L)
            if not mass:
                break
            ranked_mass = sorted(mass.items(), key=lambda x: x[1], reverse=True)
            (n1, p1) = ranked_mass[0]
            p2 = ranked_mass[1][1] if len(ranked_mass) > 1 else 0.0
            gap = p1 - p2
            trace.append((L, n1, p1, p2, gap))
            if p1 >= P1_MIN and gap >= GAP_MIN:
                chosen = n1
                continue
            decision = {"final": chosen if chosen else n1, "reason": f"ambiguity_level_{L}", "trace": trace}
            break
        else:
            decision = {"final": chosen if chosen else topM[0], "reason": "max_consensus_depth", "trace": trace}
    
    final_code = decision["final"]
    final_level = code_to_row[final_code]["level"]
    
    if verbose:
        print(f"Query: {query_text}")
        print(f"  → Predicted: {final_code} (L{final_level}) | {code_to_row[final_code]['label']}")
        print(f"  → Reason: {decision.get('reason')}")
    
    return {
        "final_code": final_code,
        "final_level": final_level,
        "reason": decision.get("reason"),
        "retrieved": retrieved[:5],
        "topM": topM[:5],
        "decision": decision
    }

print("✅ Classification function defined")


✅ Classification function defined


In [26]:
# 10c) Run Validation on training_sentences Dataset

# Configuration
NUM_VALIDATION_SAMPLES = 100  # Set to None to validate all samples
USE_BOTH_FIELDS = True  # Combine Job Description + Industry Description

# Prepare validation examples
validation_examples = []
for sent in validation_data[:NUM_VALIDATION_SAMPLES] if NUM_VALIDATION_SAMPLES else validation_data:
    # Extract query text from fields
    fields = sent.get("fields", {})
    if USE_BOTH_FIELDS:
        job_desc = fields.get("Job Description", "")
        ind_desc = fields.get("Industry Description", "")
        query_text = f"{job_desc} {ind_desc}".strip()
    else:
        query_text = fields.get("Job Description", "")
    
    if not query_text:
        continue
    
    # Extract ground truth annotations (hierarchical labels)
    annotations = sent.get("annotations", [])
    ground_truth = {ann["level"]: ann["code"] for ann in annotations}
    
    validation_examples.append({
        "sentence_id": sent["sentenceId"],
        "query": query_text,
        "ground_truth": ground_truth
    })

print(f"Prepared {len(validation_examples)} validation examples from training_sentences")
print("\nSample validation example:")
print(f"  Query: {validation_examples[0]['query']}")
print(f"  Ground truth: {validation_examples[0]['ground_truth']}")

# Run classification on all validation examples
print(f"\n{'='*80}")
print("RUNNING VALIDATION ON TRAINING_SENTENCES...")
print(f"{'='*80}\n")

results = []
for i, example in enumerate(validation_examples):
    if i > 0 and i % 10 == 0:
        print(f"Progress: {i}/{len(validation_examples)} examples classified...")
    
    prediction = classify_query(example["query"], verbose=False)
    
    # Extract predicted path (all ancestors of final prediction)
    predicted_code = prediction["final_code"]
    predicted_path = path_to_root(predicted_code, parent)
    
    # Build predicted labels at each level
    predicted_by_level = {}
    for code in predicted_path:
        level = code_to_row[code]["level"]
        predicted_by_level[level] = code
    
    results.append({
        "sentence_id": example["sentence_id"],
        "query": example["query"],
        "ground_truth": example["ground_truth"],
        "predicted": predicted_by_level,
        "final_prediction": predicted_code,
        "final_level": prediction["final_level"],
        "reason": prediction["reason"]
    })

print(f"✅ Validation complete: {len(results)} examples classified from training_sentences")


Prepared 100 validation examples from training_sentences

Sample validation example:
  Query: baby sitter caring for baby
  Ground truth: {1: '5', 2: '53', 3: '531', 4: '5311'}

RUNNING VALIDATION ON TRAINING_SENTENCES...

Progress: 10/100 examples classified...
Progress: 20/100 examples classified...
Progress: 30/100 examples classified...
Progress: 40/100 examples classified...
Progress: 50/100 examples classified...
Progress: 60/100 examples classified...
Progress: 70/100 examples classified...
Progress: 80/100 examples classified...
Progress: 90/100 examples classified...
✅ Validation complete: 100 examples classified from training_sentences


In [27]:
# 10d) Calculate Metrics for training_sentences Dataset

# Calculate accuracy at each level
max_level = max(code_to_row[c]["level"] for c in taxonomy["code"])
level_metrics = {L: {"correct": 0, "total": 0} for L in range(1, max_level + 1)}

# Track hierarchical consistency
hierarchical_correct = 0
exact_match = 0

for result in results:
    gt = result["ground_truth"]
    pred = result["predicted"]
    
    # Check accuracy at each level
    for level in range(1, max_level + 1):
        if level in gt:
            level_metrics[level]["total"] += 1
            if level in pred and pred[level] == gt[level]:
                level_metrics[level]["correct"] += 1
    
    # Check if entire path is correct (hierarchical consistency)
    all_correct = all(
        level in pred and pred[level] == gt[level]
        for level in gt.keys()
    )
    if all_correct:
        hierarchical_correct += 1
    
    # Check exact match at deepest level
    deepest_gt_level = max(gt.keys())
    deepest_gt_code = gt[deepest_gt_level]
    if result["final_prediction"] == deepest_gt_code:
        exact_match += 1

# Print results
print("\n" + "="*80)
print("VALIDATION RESULTS - TRAINING_SENTENCES DATASET")
print("="*80)
print(f"Total validation examples: {len(results)}")
print(f"\nAccuracy by Level:")
for level in range(1, max_level + 1):
    if level_metrics[level]["total"] > 0:
        acc = level_metrics[level]["correct"] / level_metrics[level]["total"]
        print(f"  Level {level}: {level_metrics[level]['correct']}/{level_metrics[level]['total']} ({acc*100:.1f}%)")

print(f"\nHierarchical Accuracy (all levels correct): {hierarchical_correct}/{len(results)} ({hierarchical_correct/len(results)*100:.1f}%)")
print(f"Exact Match (deepest level): {exact_match}/{len(results)} ({exact_match/len(results)*100:.1f}%)")

# Analyze stopping reasons
reason_counts = defaultdict(int)
for result in results:
    reason_counts[result["reason"]] += 1

print(f"\nStopping Reasons:")
for reason, count in sorted(reason_counts.items(), key=lambda x: -x[1]):
    print(f"  {reason}: {count} ({count/len(results)*100:.1f}%)")



VALIDATION RESULTS - TRAINING_SENTENCES DATASET
Total validation examples: 100

Accuracy by Level:
  Level 1: 47/100 (47.0%)
  Level 2: 11/100 (11.0%)
  Level 3: 9/100 (9.0%)
  Level 4: 7/100 (7.0%)

Hierarchical Accuracy (all levels correct): 7/100 (7.0%)
Exact Match (deepest level): 7/100 (7.0%)

Stopping Reasons:
  no_lca: 51 (51.0%)
  lca_tie_stop: 49 (49.0%)


# 11) Validation Against taxonomy_training Dataset

Now let's validate against the `taxonomy_training` dataset, which contains:
- Full taxonomy node descriptions (text from definitions)
- Ground truth labels at each level
- Hierarchical parent-child relationships

This provides a different validation scenario: classifying rich textual descriptions rather than short job+industry descriptions.

In [33]:
# 11a) Run Validation on taxonomy_training Dataset

# Configuration
NUM_TAXONOMY_SAMPLES = 300  # Set to None to validate all taxonomy nodes

# Prepare validation examples from taxonomy_training
taxonomy_validation_examples = []
for idx, row in taxonomy_training_df.iterrows():
    if NUM_TAXONOMY_SAMPLES and idx >= NUM_TAXONOMY_SAMPLES:
        break
    
    query_text = row['label']
    code = str(row['code'])
    
    # Skip if query is empty
    if not query_text or pd.isna(query_text):
        continue
    
    # Build ground truth path by walking up the taxonomy
    ground_truth = {}
    current_code = code
    while current_code in code_to_row:
        node = code_to_row[current_code]
        level = node['level']
        ground_truth[level] = current_code
        # Move to parent
        parent_code = node.get('parentCode')
        if pd.isna(parent_code) or parent_code == 'nan' or not parent_code:
            break
        current_code = str(parent_code)
    
    taxonomy_validation_examples.append({
        "code": code,
        "query": query_text,
        "ground_truth": ground_truth
    })

print(f"Prepared {len(taxonomy_validation_examples)} validation examples from taxonomy_training")
print("\nSample validation example:")
print(f"  Code: {taxonomy_validation_examples[0]['code']}")
print(f"  Query: {taxonomy_validation_examples[0]['query'][:100]}...")
print(f"  Ground truth: {taxonomy_validation_examples[0]['ground_truth']}")

# Run classification on taxonomy_training examples
print(f"\n{'='*80}")
print("RUNNING VALIDATION ON TAXONOMY_TRAINING...")
print(f"{'='*80}\n")

taxonomy_results = []
for i, example in enumerate(taxonomy_validation_examples):
    if i > 0 and i % 10 == 0:
        print(f"Progress: {i}/{len(taxonomy_validation_examples)} examples classified...")
    
    prediction = classify_query(example["query"], verbose=False)
    
    # Extract predicted path
    predicted_code = prediction["final_code"]
    predicted_path = path_to_root(predicted_code, parent)
    
    # Build predicted labels at each level
    predicted_by_level = {}
    for code in predicted_path:
        level = code_to_row[code]["level"]
        predicted_by_level[level] = code
    
    taxonomy_results.append({
        "code": example["code"],
        "query": example["query"],
        "ground_truth": example["ground_truth"],
        "predicted": predicted_by_level,
        "final_prediction": predicted_code,
        "final_level": prediction["final_level"],
        "reason": prediction["reason"]
    })

print(f"✅ Validation complete: {len(taxonomy_results)} examples classified")

Prepared 300 validation examples from taxonomy_training

Sample validation example:
  Code: 1
  Query: Managers...
  Ground truth: {1: '1'}

RUNNING VALIDATION ON TAXONOMY_TRAINING...

Progress: 10/300 examples classified...
Progress: 20/300 examples classified...
Progress: 30/300 examples classified...
Progress: 40/300 examples classified...
Progress: 50/300 examples classified...
Progress: 60/300 examples classified...
Progress: 70/300 examples classified...
Progress: 80/300 examples classified...
Progress: 90/300 examples classified...
Progress: 100/300 examples classified...
Progress: 110/300 examples classified...
Progress: 120/300 examples classified...
Progress: 130/300 examples classified...
Progress: 140/300 examples classified...
Progress: 150/300 examples classified...
Progress: 160/300 examples classified...
Progress: 170/300 examples classified...
Progress: 180/300 examples classified...
Progress: 190/300 examples classified...
Progress: 200/300 examples classified...
Prog

In [34]:
# 11b) Calculate and Report Metrics for taxonomy_training

# Calculate accuracy at each level
max_level = max(code_to_row[c]["level"] for c in taxonomy["code"])
taxonomy_level_metrics = {L: {"correct": 0, "total": 0} for L in range(1, max_level + 1)}

# Track hierarchical consistency
taxonomy_hierarchical_correct = 0
taxonomy_exact_match = 0

for result in taxonomy_results:
    gt = result["ground_truth"]
    pred = result["predicted"]
    
    # Check accuracy at each level
    for level in range(1, max_level + 1):
        if level in gt:
            taxonomy_level_metrics[level]["total"] += 1
            if level in pred and pred[level] == gt[level]:
                taxonomy_level_metrics[level]["correct"] += 1
    
    # Check if entire path is correct (hierarchical consistency)
    all_correct = all(
        level in pred and pred[level] == gt[level]
        for level in gt.keys()
    )
    if all_correct:
        taxonomy_hierarchical_correct += 1
    
    # Check exact match at deepest level
    deepest_gt_level = max(gt.keys())
    deepest_gt_code = gt[deepest_gt_level]
    if result["final_prediction"] == deepest_gt_code:
        taxonomy_exact_match += 1

# Print results
print("\n" + "="*80)
print("VALIDATION RESULTS - TAXONOMY_TRAINING DATASET")
print("="*80)
print(f"Total validation examples: {len(taxonomy_results)}")
print(f"\nAccuracy by Level:")
for level in range(1, max_level + 1):
    if taxonomy_level_metrics[level]["total"] > 0:
        acc = taxonomy_level_metrics[level]["correct"] / taxonomy_level_metrics[level]["total"]
        print(f"  Level {level}: {taxonomy_level_metrics[level]['correct']}/{taxonomy_level_metrics[level]['total']} ({acc*100:.1f}%)")

print(f"\nHierarchical Accuracy (all levels correct): {taxonomy_hierarchical_correct}/{len(taxonomy_results)} ({taxonomy_hierarchical_correct/len(taxonomy_results)*100:.1f}%)")
print(f"Exact Match (deepest level): {taxonomy_exact_match}/{len(taxonomy_results)} ({taxonomy_exact_match/len(taxonomy_results)*100:.1f}%)")

# Analyze stopping reasons
taxonomy_reason_counts = defaultdict(int)
for result in taxonomy_results:
    taxonomy_reason_counts[result["reason"]] += 1

print(f"\nStopping Reasons:")
for reason, count in sorted(taxonomy_reason_counts.items(), key=lambda x: -x[1]):
    print(f"  {reason}: {count} ({count/len(taxonomy_results)*100:.1f}%)")



VALIDATION RESULTS - TAXONOMY_TRAINING DATASET
Total validation examples: 300

Accuracy by Level:
  Level 1: 259/300 (86.3%)
  Level 2: 67/296 (22.6%)
  Level 3: 56/279 (20.1%)
  Level 4: 39/217 (18.0%)

Hierarchical Accuracy (all levels correct): 61/300 (20.3%)
Exact Match (deepest level): 42/300 (14.0%)

Stopping Reasons:
  lca_tie_stop: 212 (70.7%)
  no_lca: 88 (29.3%)


In [35]:
# 11c) Comparative Summary - Both Datasets

print("\n" + "="*80)
print("COMPARATIVE SUMMARY")
print("="*80)

# Create comparison table
print(f"\n{'Metric':<40} | {'training_sentences':>18} | {'taxonomy_training':>18}")
print("-" * 80)

# Exact match comparison
sentences_exact_pct = (exact_match / len(results) * 100) if len(results) > 0 else 0
taxonomy_exact_pct = (taxonomy_exact_match / len(taxonomy_results) * 100) if len(taxonomy_results) > 0 else 0
print(f"{'Exact Match (deepest level)':<40} | {exact_match:>6}/{len(results):<3} ({sentences_exact_pct:>5.1f}%) | {taxonomy_exact_match:>6}/{len(taxonomy_results):<3} ({taxonomy_exact_pct:>5.1f}%)")

# Hierarchical accuracy comparison
sentences_hier_pct = (hierarchical_correct / len(results) * 100) if len(results) > 0 else 0
taxonomy_hier_pct = (taxonomy_hierarchical_correct / len(taxonomy_results) * 100) if len(taxonomy_results) > 0 else 0
print(f"{'Hierarchical Accuracy (all levels)':<40} | {hierarchical_correct:>6}/{len(results):<3} ({sentences_hier_pct:>5.1f}%) | {taxonomy_hierarchical_correct:>6}/{len(taxonomy_results):<3} ({taxonomy_hier_pct:>5.1f}%)")

# Level-by-level comparison
print(f"\n{'Level-by-Level Accuracy:':<40}")
for level in range(1, max_level + 1):
    sentences_lv = level_metrics[level]
    taxonomy_lv = taxonomy_level_metrics[level]
    
    if sentences_lv["total"] > 0 or taxonomy_lv["total"] > 0:
        sent_acc = (sentences_lv["correct"] / sentences_lv["total"] * 100) if sentences_lv["total"] > 0 else 0
        taxo_acc = (taxonomy_lv["correct"] / taxonomy_lv["total"] * 100) if taxonomy_lv["total"] > 0 else 0
        
        print(f"  {'Level ' + str(level):<38} | {sentences_lv['correct']:>6}/{sentences_lv['total']:<3} ({sent_acc:>5.1f}%) | {taxonomy_lv['correct']:>6}/{taxonomy_lv['total']:<3} ({taxo_acc:>5.1f}%)")

print("\n" + "="*80)
print("DATASET CHARACTERISTICS")
print("="*80)
print(f"\ntraining_sentences:")
print(f"  - Short job + industry descriptions")
print(f"  - Real-world query patterns")
print(f"  - {len(results)} examples validated")

print(f"\ntaxonomy_training:")
print(f"  - Full taxonomy node definitions")
print(f"  - Rich semantic descriptions")
print(f"  - {len(taxonomy_results)} examples validated")

print(f"\n{'='*80}")
print("KEY INSIGHTS")
print("="*80)

# Calculate improvement/degradation
exact_diff = taxonomy_exact_pct - sentences_exact_pct
hier_diff = taxonomy_hier_pct - sentences_hier_pct

if exact_diff > 5:
    print(f"\n✓ Algorithm performs {exact_diff:.1f}% better on taxonomy definitions")
    print(f"  → Multi-view embeddings effectively leverage rich semantic content")
elif exact_diff < -5:
    print(f"\n✗ Algorithm performs {abs(exact_diff):.1f}% worse on taxonomy definitions")
    print(f"  → May need tuning for longer text passages")
else:
    print(f"\n≈ Similar performance across both datasets (within {abs(exact_diff):.1f}%)")
    print(f"  → Algorithm is robust across different query types")



COMPARATIVE SUMMARY

Metric                                   | training_sentences |  taxonomy_training
--------------------------------------------------------------------------------
Exact Match (deepest level)              |      7/100 (  7.0%) |     42/300 ( 14.0%)
Hierarchical Accuracy (all levels)       |      7/100 (  7.0%) |     61/300 ( 20.3%)

Level-by-Level Accuracy:                
  Level 1                                |     47/100 ( 47.0%) |    259/300 ( 86.3%)
  Level 2                                |     11/100 ( 11.0%) |     67/296 ( 22.6%)
  Level 3                                |      9/100 (  9.0%) |     56/279 ( 20.1%)
  Level 4                                |      7/100 (  7.0%) |     39/217 ( 18.0%)

DATASET CHARACTERISTICS

training_sentences:
  - Short job + industry descriptions
  - Real-world query patterns
  - 100 examples validated

taxonomy_training:
  - Full taxonomy node definitions
  - Rich semantic descriptions
  - 300 examples validated

KEY INS

In [43]:
# 11d) Error Analysis - taxonomy_training Dataset

# Find misclassified examples
taxonomy_errors = []
for result in taxonomy_results:
    deepest_gt_level = max(result["ground_truth"].keys())
    deepest_gt_code = result["ground_truth"][deepest_gt_level]
    
    if result["final_prediction"] != deepest_gt_code:
        # Determine which level the error occurred at
        error_level = None
        for level in range(1, max_level + 1):
            if level in result["ground_truth"]:
                gt_code = result["ground_truth"][level]
                pred_code = result["predicted"].get(level, None)
                if pred_code != gt_code:
                    error_level = level
                    break
        
        taxonomy_errors.append({
            **result,
            "error_level": error_level
        })

print(f"\n{'='*80}")
print(f"ERROR ANALYSIS - TAXONOMY_TRAINING ({min(10, len(taxonomy_errors))} examples)")
print(f"{'='*80}\n")

for i, err in enumerate(taxonomy_errors[:10]):
    print(f"Example {i+1}:")
    print(f"  Code: {err['code']}")
    print(f"  Query: {err['query']}")
    print(f"  Ground Truth Path:")
    for level in sorted(err["ground_truth"].keys()):
        code = err["ground_truth"][level]
        label = code_to_row[code]["label"]
        print(f"    L{level}: {code} | {label}")
    print(f"  Predicted Path:")
    for level in sorted(err["predicted"].keys()):
        code = err["predicted"][level]
        label = code_to_row[code]["label"]
        marker = " ✗" if level in err["ground_truth"] and err["predicted"][level] != err["ground_truth"][level] else ""
        print(f"    L{level}: {code} | {label}{marker}")
    print(f"  Error first occurred at Level {err['error_level']}")
    print(f"  Stopping reason: {err['reason']}")
    print()

# Summary statistics
taxonomy_error_by_level = defaultdict(int)
for err in taxonomy_errors:
    if err["error_level"]:
        taxonomy_error_by_level[err["error_level"]] += 1

if len(taxonomy_errors) > 0:
    print(f"{'='*80}")
    print("ERROR DISTRIBUTION BY LEVEL - TAXONOMY_TRAINING")
    print(f"{'='*80}")
    for level in sorted(taxonomy_error_by_level.keys()):
        pct = taxonomy_error_by_level[level]/len(taxonomy_errors)*100
        print(f"  Level {level}: {taxonomy_error_by_level[level]} errors ({pct:.1f}% of all errors)")
else:
    print(f"{'='*80}")
    print("✅ NO ERRORS - Perfect classification on all taxonomy_training examples!")
    print(f"{'='*80}")



ERROR ANALYSIS - TAXONOMY_TRAINING (10 examples)

Example 1:
  Code: 11
  Query: Chief Executives, Senior Officials and Legislators
  Ground Truth Path:
    L1: 1 | Managers
    L2: 11 | Chief Executives, Senior Officials and Legislators
  Predicted Path:
    L1: 1 | Managers
    L2: 11 | Chief Executives, Senior Officials and Legislators
    L3: 111 | Legislators and Senior Officials
    L4: 1112 | Senior Government Officials
  Error first occurred at Level None
  Stopping reason: no_lca

Example 2:
  Code: 111
  Query: Legislators and Senior Officials
  Ground Truth Path:
    L1: 1 | Managers
    L2: 11 | Chief Executives, Senior Officials and Legislators
    L3: 111 | Legislators and Senior Officials
  Predicted Path:
    L1: 1 | Managers
    L2: 11 | Chief Executives, Senior Officials and Legislators
    L3: 111 | Legislators and Senior Officials
    L4: 1112 | Senior Government Officials
  Error first occurred at Level None
  Stopping reason: no_lca

Example 3:
  Code: 1111
  Que